In [1]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

def create_embedding(texts):
    if type(texts) == list: # the input param is a list contains text chunks
        embeddings = model.encode(texts)
        return embeddings
    elif type(texts) == str: # just one text chunk
        embedding = model.encode([texts])
        return embedding
    else:
        raise ValueError("The texts paramters should be str or list[str]")

print(device)

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: d:\Documents\GitHub\NodeRAG\testing
The root directory is: d:\Documents\GitHub\NodeRAG


In [3]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response.text, response.usage_metadata.total_token_count

In [4]:
import sys
import numpy as np
sys.path.append(root_path)
from testing.metrics.faithfulness import compute_faithfulness #Detect hallucinations: Measures how much the LLM answer is based on the retrieved context
"""
question: str,
answer: str,
contexts: List[str],
call_gemini,
max_retries: int = 5
"""
from testing.metrics.context_recall import compute_context_recall #Evaluate retrieval coverage: Measures how much of the reference (ground-truth) answer is supported by the retrieved context.
"""
question: str,
contexts: List[str],
reference_answer: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.coverage import compute_coverage #Measures how completely a model’s response covers the factual content of a reference (ground-truth) answer.
"""
question: str,
reference: str,
response: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.rouge import compute_rouge
#Calculate F-1 of lexical overlap by LCS: longest common subsequence (in terms of word order, adjacency not required)
#Precision = len(LCS) / len(Answer)
#Recall = len(LCS) / len(Ref)
"""
answer: str,
ground_truth: str,
rouge_type: str = "rougeL",
mode: str = "fmeasure"
"""
from testing.metrics.accuracy import compute_answer_accuracy
"""
question: str,
answer: str,
ground_truth: str,
call_gemini,
create_embedding,
weights: List[float] = [0.75, 0.25],
beta: float = 1.0,
max_retries: int = 5
"""
print("Import evaluation functions")


Import evaluation functions


In [5]:
import pandas as pd
medical_questions_answered_loaded = pd.read_parquet("data/medical_questions_answered.parquet")
relevant_questions = medical_questions_answered_loaded[medical_questions_answered_loaded["question_type"] == "Contextual Summarize"]
question_ids = relevant_questions['id'].to_list()
relevant_questions

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
1607,Medical-5fd3c02d,Medical,What are the main risk factors associated with...,The main risk factors associated with the deve...,Contextual Summarize,Basal cell carcinoma risk is increased by UV r...,UV radiation exposure is a primary risk factor...,The main risk factors for developing basal cel...,Basal cell skin cancer is caused by exposure t...,8096.0
1608,Medical-b8b22d33,Medical,Which anatomical locations and cell types are ...,Basal cell carcinoma arises from basal cells i...,Contextual Summarize,Basal cell carcinoma arises from basal cells i...,BCC arises from basal cells in the lower part ...,Basal cell carcinoma (BCC) most commonly invol...,Basal cell skin cancer is the most common type...,5781.0
1609,Medical-9c85110b,Medical,What are the typical symptoms and presentation...,Basal cell carcinoma (BCC) typically presents ...,Contextual Summarize,Basal cell carcinoma typically presents as fla...,"BCC presents as flat, pale or yellow areas, re...",Basal cell carcinoma (BCC) typically appears i...,Basal cell skin cancer is the most common type...,6137.0
1610,Medical-fcf87025,Medical,Which diagnostic methods are used to confirm b...,Diagnosis of basal cell carcinoma involves a t...,Contextual Summarize,Diagnosis of basal cell carcinoma involves a c...,Diagnosis of BCC involves medical and family h...,The diagnostic process for basal cell carcinom...,A diagnosis of bone cancer is confirmed using ...,4795.0
1611,Medical-6baf8bff,Medical,What are the standard treatment options for ba...,The standard treatment options for basal cell ...,Contextual Summarize,Surgery is the most common treatment for basal...,Surgery is the most common treatment for BCC; ...,Standard treatment options for basal cell carc...,Standard surgical excision is a recommended tr...,20750.0
...,...,...,...,...,...,...,...,...,...,...
1891,Medical-0cba5d0a,Medical,What are the main risk factors and considerati...,The main risk factors for breast cancer includ...,Contextual Summarize,Major risk factors for breast cancer include f...,"Risk factors include family history, BRCA muta...",Key risk factors for breast cancer include bei...,Questions regarding cancer treatment options c...,10341.0
1892,Medical-d72d7cb9,Medical,How is follow-up and surveillance conducted fo...,Follow-up for breast cancer patients includes ...,Contextual Summarize,Follow-up for breast cancer patients includes ...,"Follow-up includes imaging, physical exams, an...","I am sorry, but the provided context does not ...","After treatment, patients undergo surveillance...",27041.0
1893,Medical-97d64c1f,Medical,What is the clinical significance of tumor gra...,The clinical significance of tumor grade in br...,Contextual Summarize,Tumor grade in breast cancer is described as G...,Grade describes how abnormal the tumor cells l...,Tumor grade in breast cancer is clinically sig...,Cancer grades describe how quickly cancers gro...,6771.0
1894,Medical-4dc06fbd,Medical,What are the most common therapies used in the...,Systemic therapies for breast cancer include c...,Contextual Summarize,Systemic therapies for breast cancer include c...,Systemic therapy is used for invasive and meta...,The most common systemic therapies used in bre...,Most types of radiation include several short ...,13941.0


In [6]:
try:
    medical_questions_evaluated = pd.read_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
    medical_questions_evaluated = (
        medical_questions_evaluated
        .set_index("qid")
        .to_dict(orient="index")
    )
except:
    medical_questions_evaluated = {}


medical_questions_evaluated

{'Medical-5fd3c02d': {'accuracy': 0.8975624233072602,
  'coverage': 0.8,
  'tokens': 12639},
 'Medical-b8b22d33': {'accuracy': 0.799239873834546,
  'coverage': 0.6666666666666666,
  'tokens': 6866},
 'Medical-9c85110b': {'accuracy': 0.856498707381742,
  'coverage': 0.7857142857142857,
  'tokens': 10808}}

In [7]:
def all_valid(values):
    return all(
        v is not None and not np.isnan(v)
        for v in values
    )

In [8]:
#accuracy
#coverage

from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 10
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in medical_questions_evaluated:
            continue

        row = relevant_questions.loc[relevant_questions["id"] == qid].iloc[0]
        row_eval = {}

        question = row["question"]
        reference = row["answer"]
        answer = row["LLM_answer"]
        context = row["LLM_context"]
        contexts = context.split(sep)
        
        accuracy, token1 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                accuracy, token1 = compute_answer_accuracy(question, answer, reference, call_gemini, create_embedding)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} accuracy calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

        coverage, token2 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                coverage, token2 = compute_coverage(question, reference, answer, call_gemini)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} coverage calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff
        if all_valid([accuracy, coverage, token1, token2]):
            row_eval["accuracy"] = accuracy
            row_eval["coverage"] = coverage
            row_eval["tokens"] = token1 + token2
            medical_questions_evaluated[qid] = row_eval
get_answers()


100%|██████████| 289/289 [3:43:00<00:00, 46.30s/it]  


In [9]:
medical_questions_evaluated

{'Medical-5fd3c02d': {'accuracy': 0.8975624233072602,
  'coverage': 0.8,
  'tokens': 12639},
 'Medical-b8b22d33': {'accuracy': 0.799239873834546,
  'coverage': 0.6666666666666666,
  'tokens': 6866},
 'Medical-9c85110b': {'accuracy': 0.856498707381742,
  'coverage': 0.7857142857142857,
  'tokens': 10808},
 'Medical-fcf87025': {'accuracy': 0.6135058700694611,
  'coverage': 0.8,
  'tokens': 9777},
 'Medical-6baf8bff': {'accuracy': 0.6305920222156903,
  'coverage': 0.21052631578947367,
  'tokens': 10218},
 'Medical-643b2b9d': {'accuracy': 0.6099826097019654,
  'coverage': 1.0,
  'tokens': 7927},
 'Medical-25f8ca5e': {'accuracy': 0.5365685104843791,
  'coverage': 0.5,
  'tokens': 6797},
 'Medical-d961b796': {'accuracy': 0.7094433605243194,
  'coverage': 0.8,
  'tokens': 10810},
 'Medical-915df475': {'accuracy': 0.7335145324028612,
  'coverage': 0.625,
  'tokens': 12166},
 'Medical-e0e8fd6d': {'accuracy': 0.8427452802240482,
  'coverage': 0.8571428571428571,
  'tokens': 9904},
 'Medical-d701

In [10]:
df_eval = (
    pd.DataFrame.from_dict(medical_questions_evaluated, orient="index")
      .reset_index()
      .rename(columns={"index": "qid"})
)

In [11]:
df_eval.to_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
df_eval_loaded = pd.read_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
df_eval_loaded

,qid,accuracy,coverage,tokens
0,Medical-5fd3c02d,0.897562,0.800000,12639
1,Medical-b8b22d33,0.799240,0.666667,6866
2,Medical-9c85110b,0.856499,0.785714,10808
3,Medical-fcf87025,0.613506,0.800000,9777
4,Medical-6baf8bff,0.630592,0.210526,10218
...,...,...,...,...
284,Medical-0cba5d0a,0.604894,0.583333,16398
285,Medical-d72d7cb9,0.400845,0.000000,6130
286,Medical-97d64c1f,0.863777,0.750000,11560
287,Medical-4dc06fbd,0.581375,0.666667,9562


In [12]:
print("min:", df_eval_loaded["tokens"].min())
print("max:", df_eval_loaded["tokens"].max())
print("avg:", df_eval_loaded["tokens"].mean())
print("sum:", df_eval_loaded["tokens"].sum())

min: 2650
max: 24404
avg: 10683.259515570935
sum: 3087462


In [13]:
print("min:", df_eval_loaded["accuracy"].min())
print("max:", df_eval_loaded["accuracy"].max())
print("avg:", df_eval_loaded["accuracy"].mean())
print("sum:", df_eval_loaded["accuracy"].sum())

min: 0.21056970953941345
max: 0.9956524967397278
avg: 0.662963362888527
sum: 191.5964118747843


In [14]:
print("min:", df_eval_loaded["coverage"].min())
print("max:", df_eval_loaded["coverage"].max())
print("avg:", df_eval_loaded["coverage"].mean())
print("sum:", df_eval_loaded["coverage"].sum())

min: 0.0
max: 1.0
avg: 0.5918892969240213
sum: 171.05600681104215
